# SmishGuard — one-click fixed pipeline

**Runtime → Change runtime type → T4 GPU.** Then just **Run all** (top to bottom). It:

1. pulls the latest fixed scripts from GitHub,
2. rebuilds `data/` (corrected labels + curated seeds),
3. retrains (saves best epoch by F1),
4. exports int8 ONNX with a **parity gate** — fails loudly instead of shipping a broken model,
5. evaluates honestly,
6. uploads to a **new** model repo so there is no stale cache or "no files modified" problem.

Set `HF_USERNAME` and paste a **Write token** in the upload cell below.

In [ ]:
!pip -q install "transformers>=4.44" datasets onnx onnxruntime scikit-learn accelerate

In [ ]:
import os

REPO_URL = "https://github.com/Ricky-zzz/smishguard.git"

if os.path.isdir("/content/smishguard/.git"):
    !git -C /content/smishguard fetch -q origin && git -C /content/smishguard reset --hard -q origin/main
else:
    !git clone -q $REPO_URL /content/smishguard

os.chdir("/content/smishguard/training")
print("On commit:", !git -C /content/smishguard log --oneline -1)

## 1. Data (downloads PH sets + seeds)

In [ ]:
!python prepare_data.py --ph-preset --bootstrap --out data

## 2. Fine-tune (saves best epoch by F1)

In [ ]:
!python finetune.py --model sentence-transformers/all-MiniLM-L6-v2 --data data --out model --epochs 5

## 3. Export int8 ONNX (parity-gated — must say `Parity check passed.`)

In [ ]:
!python export_onnx.py --model-dir model --out onnx_out
!find onnx_out -maxdepth 2 -type f | sort

## 4. Evaluate honestly

In [ ]:
!python eval.py --model-dir model --data data/test.csv

## 5. Upload to a NEW model repo (no stale cache)

Create a **Write** token at https://huggingface.co/settings/tokens. Set `HF_USERNAME` and run.

In [ ]:
HF_USERNAME = ""  # <-- your Hugging Face username
assert HF_USERNAME, "Set HF_USERNAME first."

from huggingface_hub import HfApi, login, whoami

login()
print("Logged in as:", whoami()["name"])

NEW_REPO = f"{HF_USERNAME}/smishguard-minilm-v2"
api = HfApi()
api.create_repo(NEW_REPO, repo_type="model", exist_ok=True)
api.upload_folder(folder_path="onnx_out", repo_id=NEW_REPO, repo_type="model")
print("Uploaded to https://huggingface.co/" + NEW_REPO)

## 6. Sanity check (optional) — the messages you will demo

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

clf = pipeline("text-classification", model="model", tokenizer=AutoTokenizer.from_pretrained("model"), top_k=None)
for t in [
    "BPI: Your transaction of PHP 250.00 at SM MAKATI was posted. If this was not you, call 889-10000. Never share your OTP.",
    "Shopee: Your order #12345678 is out for delivery today. Rider: Juan D. Estimated time: 2-5 PM.",
    "Nakuha ko na po yung padala niyo. Salamat po! - Aling Nena",
    "GCash: Na-block ang account mo. I-verify agad: https://gcash-verify.top/otp at ilagay ang 6-digit code.",
]:
    r = max(clf(t, truncation=True, max_length=128)[0], key=lambda d: d["score"])
    print(r["label"], f"{r['score']:.3f}", "|", t[:45])